# TRAILS: Simple Numerical Example (Tutorial)

**Authors**: Romain Sacchi (PSI)

**Contact**: romain.sacchi@psi.ch

## Purpose

This notebook is a guided, end-to-end first experience with TRAILS. It covers:

1. Loading a small example data package.
2. Exploring the technosphere (`A`) and biosphere (`B`) matrices.
3. Temporal routing and graph visualization.
4. Running temporal and static LCA, then plotting results.
5. Linking results to the FaIR climate emulator (radiative forcing + temperature).
## Theory (short)

TRAILS (Temporal Routing And Aggregation of Impacts across Life-cycle Systems)
extends classic LCA by making time an explicit dimension. Exchanges can carry
**temporal distributions** (discrete, normal, lognormal, uniform, triangular),
which are expanded into year offsets during traversal. For each calendar year
that becomes active, the system is solved and biosphere flows are accumulated
at their respective years. This yields **time-resolved inventories** and **impact
scores** that answer questions like *when impacts occur* and *which upstream
suppliers dominate over time*.

**Environment requirements (from `pyproject.toml`):**
- Python `>=3.10, <3.13`
- Dependencies are loaded from `requirements.txt`
- Optional extras: `testing` (pytest), `docs` (sphinx)


In [1]:
# Core imports
from pathlib import Path
from datapackage import Package

from trails import (
    Trails,
    get_lcia_method_names,
    plot_temporal_scores,
    plot_rf,
    plot_temp,
    clear_cache,
)

## 1. Load the example data package

We use the small data package shipped in `examples/example data package`.


**Caching note**: the first time you load a data package, TRAILS builds
        and caches the technosphere and biosphere matrices (and their indices).
        Subsequent runs reuse the cache for faster startup. If you change the
        data package or want a clean rebuild, use `clear_cache()` to remove the
        cached matrices and force regeneration on the next load.


In [2]:
# Point to the example data package
package_path = Path('example data package/datapackage.json')
package = Package(str(package_path))

# Initialize Trails
trails = Trails(package)

Loading matrices from data package          [1/4]
Loading indices from data package           [2/4]
Interpolating matrices to annual resolution [3/4]
Building cache                              [4/4]
Data package cached at: /Users/romain/Library/Application Support/trails/cache/interp_592d90d85ab3


In [19]:
# Optional: clear cache to force rebuilding matrices on next load
clear_cache()

PosixPath('/Users/romain/Library/Application Support/trails/cache')

## 2. Explore matrices and indices (A, B)


TRAILS exposes the technosphere matrix `A` and biosphere matrix `B` as 3D arrays:

- `A` indexed by `(year, activity, activity)`
- `B` indexed by `(year, activity, flow)`

You can inspect shapes and a few entries to get a feel for the data.

In [3]:
# Inspect matrix shapes (year, activity, activity) and (year, activity, flow)
print('A shape:', trails.A.shape)
print('B shape:', trails.B.shape)

A shape: (96, 17, 17)
B shape: (96, 17, 2)


`A` stores production and technosphere-to-technosphere exchanges, while `B` stored technosphere-to-biosphere exchanges. 

These are `sparse.COO` matrices.

In [8]:
trails.A.coords

array([[ 0,  0,  0, ..., 95, 95, 95],
       [ 0,  1,  1, ..., 15, 16, 16],
       [ 0,  1,  5, ..., 15, 15, 16]], shape=(3, 3262))

Let's check the value of the production exchange of activity `0` at year `0`

In [9]:
trails.A[
    0, # Year 0 or 2005
    0, # Activity 0
    0  # Product 0
]

np.float32(1.0)

Let's check the inputs of activity `13` at year 45 (2050).

In [18]:
trails.A[
    45, # Year 0 or 2005
    13, # Activity 0
    :  # all
].data

array([-5.e-06, -2.e-02, -5.e-06,  1.e+00, -2.e-02], dtype=float32)

In principle, we should be seeing the same here:

In [19]:
# Print exchanges for a chosen activity index (example: 13)
# Replace 13 with an id from search_activity if desired.
trails.print_exchange_table(year=2050, act_idx=13)

Temporal distribution codes:
+------+----------------------------+
| code |        distribution        |
+------+----------------------------+
|  1   | discrete (all mass at loc) |
|  2   |         lognormal          |
|  3   |           normal           |
|  4   |          uniform           |
|  5   |         triangular         |
+------+----------------------------+
Temporal distribution fields:
+------------------------+---------------------------------------+
|         field          |                meaning                |
+------------------------+---------------------------------------+
| temporal_distribution  |           distribution code           |
|      temporal_loc      | location parameter (mean/median/mode) |
|     temporal_scale     |     scale parameter (stddev/sigma)    |
|      temporal_min      |   minimum integer offset (inclusive)  |
|      temporal_max      |   maximum integer offset (inclusive)  |
| temporal_amount_source |         ported value or matrix      

You can also search activities by name and inspect exchanges for a
        specific activity.


In [13]:
from trails import search_activity

# Find activity ids by keyword
search_activity(trails, name='electricity')[:5]

index,name,reference product,location
1,"electricity, medium voltage",electricity,RER
5,natural gas electricity,electricity,RER
7,hydro electricity,electricity,RER
9,wind electricity,electricity,RER


## 3. Choose an LCIA method

TRAILS exposes a helper that lists all available LCIA methods in the package.


In [20]:
methods = get_lcia_method_names(trails)
methods[:5]  # show a few

['CML v4.8 2016 no LT - acidification no LT - acidification (incl. fate, average Europe total, A&B) no LT',
 'CML v4.8 2016 no LT - climate change no LT - global warming potential (GWP100) no LT',
 'CML v4.8 2016 no LT - ecotoxicity: freshwater no LT - freshwater aquatic ecotoxicity (FAETP inf) no LT',
 'CML v4.8 2016 no LT - ecotoxicity: marine no LT - marine aquatic ecotoxicity (MAETP inf) no LT',
 'CML v4.8 2016 no LT - ecotoxicity: terrestrial no LT - terrestrial ecotoxicity (TETP inf) no LT']

Pick a method and keep it in a single-element list (the LCA runner expects a list).

In [21]:
method = methods[:2]
method

['CML v4.8 2016 no LT - acidification no LT - acidification (incl. fate, average Europe total, A&B) no LT',
 'CML v4.8 2016 no LT - climate change no LT - global warming potential (GWP100) no LT']

## 4. Run the temporal LCA

This computes time-resolved scores and inventory. We also keep the inventory so we
can later use the climate emulator.


### 4.1 Temporal routing (graph traversal)

This step constructs a temporal routing graph from a starting activity.
It is useful for understanding time-dependent dependencies before solving.

In [24]:
# Example temporal routing (graph traversal)
trails.temporal_routing(
    start_year=2050,
    start_act_idx=13, # activity id
    amount=1.0, # function unit amount
    max_depth=3, # supply chain levels to traverse up to
)

Temporal routing: 1259node [00:00, 62874.36node/s]


You can visualize the routing graph with `plot_temporal_graph`.
It is saved as an HTML in the same folder as this notebook.


In [25]:
from trails.plotting import plot_temporal_graph

plot_temporal_graph(
    trails,
    filename='trails_graph.html',
)

'trails_graph.html'

### 4.2 Temporal LCA (full solve)

This computes time-resolved scores and inventory. We keep the inventory for the climate emulator later on. If you do not intend to use the emulator, you can probably disable inventory storage (faster). The higher `max_depth` in `temporal_routing()`, the more likely it is that the number of years to iterate increases, though.

In [27]:
trails.lca(
    methods=method,
    show_progress=True,
    compute_score=True,
    store_inventory=True,
)

Temporal LCA: solve years: 100%|█████████████| 41/41 [00:00<00:00, 165.77year/s]


We now have access to `trails.inventory` and `trails.characterized_inventory`.
For `trails.inventory`, we have amounts emitted:

* activity
* elementary flow
* year
* root activity, which is the first-level activity at teh root of all upstream emissionss (useful for plotting)

In [30]:
trails.inventory.coords

Coordinates:
  * activity       (activity) int64 136B 0 1 2 3 4 5 6 ... 10 11 12 13 14 15 16
  * flow           (flow) int64 16B 0 1
  * year           (year) int64 5kB 1997 1998 1999 2000 ... 2605 2606 2607 2608
  * root activity  (root activity) int64 136B 0 1 2 3 4 5 ... 11 12 13 14 15 16

For `trails.characterized_inventory`, it is similar, but it stores instead characterized results, with one additional dimension:
* method

In [31]:
trails.characterized_inventory.coords

Coordinates:
  * method         (method) object 16B 'CML v4.8 2016 no LT - acidification n...
  * activity       (activity) int64 136B 0 1 2 3 4 5 6 ... 10 11 12 13 14 15 16
  * flow           (flow) int64 16B 0 1
  * year           (year) int64 5kB 1997 1998 1999 2000 ... 2605 2606 2607 2608
  * root activity  (root activity) int64 136B 0 1 2 3 4 5 ... 11 12 13 14 15 16

### 4.3 Static LCA (single year)

For convenience, this provides a quick single-year check for a chosen activity.


In [33]:
# Example static LCA for a specific activity index
# Replace act_idx with an index of interest
trails.static_lca(year=2050, act_idx=13, methods=method)
trails.static_score

{'CML v4.8 2016 no LT - acidification no LT - acidification (incl. fate, average Europe total, A&B) no LT': 0.0,
 'CML v4.8 2016 no LT - climate change no LT - global warming potential (GWP100) no LT': 0.13936696134985224}

## 5. Plot temporal impacts

The default plot highlights the top contributors over time.


In [36]:
from trails.plotting import plot_temporal_scores
figs = plot_temporal_scores(
    stacked=False,
    legend_top_n=7,
    show_flow_contributions=False,
    trails=trails,
    title="",
    method_label="kg CO₂-eq",
    cumulative=False,
    width=550,
    height=450,
    year_tick=5,
    year_range=(2000, 2100),
    reference_year=2050,
    show_cumulative_axis=True,
    static_score=trails.static_score,
    static_score_dash= "dot",
    static_score_color= "red",
)

ValueError: Multiple static scores provided; pass method=... to select one.

In [ ]:
figs[0]

In [ ]:
figs[1]

## 6. Climate emulator (FaIR)

TRAILS can translate inventory time series into **radiative forcing** and
**temperature anomaly** using the FaIR climate model.
Outputs are stored as quantiles across all FaIR configurations:
`2.5, 25, 50, 75, 97.5`.


In [ ]:
from trails.fair_rf import run_fair_delta_rf

rf = run_fair_delta_rf(
    trails,
    scenario='high-extension',
    scale_target_fraction=0.1,
)

In [ ]:
# Access stored results
trails.instant_radiative_forcing
trails.delta_temperature

### Plot radiative forcing (median + quantile band)

In [ ]:
fig_rf = plot_rf(
    trails,
    by='flow',
    year_range=(2000, 2100),
    year_tick=10,
    reference_year=2050,
)
fig_rf

### Plot temperature anomaly (median + quantile band)

In [ ]:
fig_temp = plot_temp(
    trails,
    by='root activity',
    year_range=(2000, 2100),
    year_tick=10,
)
fig_temp

## 8. Next steps

- Import user inventories (Excel) with `trails.import_excel_inventory`
- Run additional LCIA methods and compare results
- Customize plots (stacked vs non-stacked, flow grouping, etc.)
